#  Preprocessing Pipeline Notebook

This notebook builds a **general, reusable preprocessing pipeline** for datasets.
It ensures raw data is consistently transformed into clean train/test sets ready for machine learning.

### Workflow Overview
1. **Load dataset** -> `load()` function supports CSV, JSON, and Excel formats.
2. **Preprocess data** -> `preprocess_data()` scales numeric columns, encodes categorical columns, separates target, and splits into train/test sets.
3. **Validation checks** -> `validation_report()` confirms shapes, target distribution, and scaling ranges.
4. **Config-driven** -> pipeline behavior is controlled by a flexible `config` dictionary (scaling method, columns, target, split size).
5. **Outputs** -> clean `X_train, X_test, y_train, y_test` sets, optionally saved for modeling.

This notebook provides a **stable, modular pipeline** that can handle multiple file formats and produce validated datasets for end-to-end testing.


In [2]:
import pandas as pd
import json

def load(file_path):
    """
    Loads a dataset from CSV, JSON, or Excel into a pandas DataFrame.

    Parameters:
        file_path (str): Path to the dataset file

    Returns:
        DataFrame: Loaded dataset
    """
    if file_path.endswith(".csv"):
        df = pd.read_csv(file_path)
    elif file_path.endswith(".json"):
        with open(file_path, "r") as f:
            data = json.load(f)
        df = pd.DataFrame(data)
    elif file_path.endswith(".xlsx") or file_path.endswith(".xls"):
        df = pd.read_excel(file_path)
    else:
        raise ValueError(f"Unsupported file format: {file_path}. Please use CSV, JSON, or Excel.")

    print(f"Loaded dataset with shape {df.shape} from {file_path}")


    return df



The `load()` function reads a dataset file and converts it into a **pandas DataFrame** for further preprocessing.

- **CSV files** -> loaded using `pd.read_csv()`
- **JSON files** -> opened with `json.load()` and converted to DataFrame
- **Excel files (.xlsx / .xls)** -> loaded using `pd.read_excel()`
- **Error handling** -> raises a clear message if the format is unsupported

 `load()` is the **entry point** of the pipeline.
It ensures that no matter the dataset format (CSV, JSON, Excel), you always get a clean DataFrame ready for preprocessing.


In [3]:
# Define Preprocessing Pipeline Function

from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split

def preprocess_data(df, config):
    """
    Preprocesses the dataset based on configuration.

    Parameters:
        df (DataFrame): Input dataset
        config (dict): Configuration dictionary with keys:
            - scale_cols: list of numeric columns to scale
            - encode_cols: list of categorical columns to encode
            - target_col: target column name
            - test_size: fraction for test split (default 0.2)
            - stratify: whether to stratify split (default True)
            - scaling: 'zscore' or 'minmax'

    Returns:
        X_train, X_test, y_train, y_test
    """

    # Copy dataset
    data = df.copy()

    # Missing value handling
    data = data.dropna()   # or data.fillna(0) depending on strategy


    # Scaling
    if config.get("scaling") == "zscore":
        scaler = StandardScaler()
    else:
        scaler = MinMaxScaler()

    data[config["scale_cols"]] = scaler.fit_transform(data[config["scale_cols"]])

    # Encoding categorical columns
    for col in config["encode_cols"]:
        le = LabelEncoder()
        data[col] = le.fit_transform(data[col])

    # Split features and target
    X = data.drop(columns=[config["target_col"]])
    y = data[config["target_col"]]

    # Train/test split
    stratify = y if config.get("stratify", True) else None
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=config.get("test_size", 0.2), stratify=stratify, random_state=42
    )

    print(f"Preprocessed dataset: {data.shape} rows, {data.shape[1]} columns")

    return X_train, X_test, y_train, y_test


The `preprocess_data()` function prepares the dataset for modeling based on the configuration provided.

- **Missing value handling** -> drops rows with null values using `dropna()` to ensure the pipeline runs smoothly.
- **Scaling** -> applies either Z‑score (`StandardScaler`) or Min‑Max (`MinMaxScaler`) to numeric columns listed in `scale_cols`.
- **Encoding** -> converts categorical columns listed in `encode_cols` into numeric values using `LabelEncoder`.
- **Target separation** -> splits the dataset into features (`X`) and target (`y`) based on `target_col`.
- **Train/Test split** -> divides the data into training and testing sets, preserving class distribution if `stratify=True`.

`preprocess_data()` transforms raw data into **clean, scaled, encoded, and split datasets** (`X_train, X_test, y_train, y_test`) ready for machine learning models.


In [4]:
configs = {
    "csv": {
            "scale_cols": ["frp"],          # numeric column to scale
            "encode_cols": ["satellite"],   # categorical column to encode
            "target_col": "confidence",     # target variable
            "test_size": 0.2,               # 80/20 split
            "stratify": True,               # preserve class distribution
            "scaling": "minmax"             # choose 'minmax' or 'zscore'
    },
    "json": {
        "scale_cols": ["temperature", "humidity"],
        "encode_cols": ["region", "sensor_type"],
        "target_col": "alert_level",
        "test_size": 0.3,
        "stratify": True,
        "scaling": "zscore"
    }
}



The `config` dictionary defines how the preprocessing pipeline should handle the dataset.

- **scale_cols** -> numeric columns to scale (here, `frp`)
- **encode_cols** -> categorical columns to encode (here, `satellite`)
- **target_col** -> the column to predict (here, `confidence`)
- **test_size** -> fraction of data reserved for testing (0.2 = 20%)
- **stratify** -> ensures train/test split preserves target distribution
- **scaling** -> method for scaling numeric values (`minmax` or `zscore`)

`config` makes the pipeline **flexible and reusable**.
By changing values in this dictionary, the same notebook can preprocess different datasets without rewriting code.


In [5]:
def get_config(file_path, configs):
    if file_path.endswith(".csv"):
        return configs["csv"]
    elif file_path.endswith(".json"):
        return configs["json"]
    elif file_path.endswith(".xlsx") or file_path.endswith(".xls"):
        return configs["csv"]  # If Excel has the same schema, reuse CSV config
    else:
        raise ValueError("Unsupported format")


# Unified Configuration Setup

Our pipeline can handle **CSV, Excel, and JSON files under one roof**.
The `load()` function reads the dataset, and the `get_config()` function automatically picks the right configuration based on the file type.

- **CSV files** -> use the `csv` config
- **Excel files (.xlsx / .xls)** -> use the `excel` config
  (if the schema is the same as CSV, we can reuse the same config)
- **JSON files** -> use the `json` config

This way:
- The pipeline code (`load()` + `preprocess_data()`) stays the same.
- Only the configuration changes depending on the dataset format.
- No manual switching is needed , the system auto‑detects the file type and applies the right config.

 Result: One clean, reusable pipeline that works across multiple dataset formats.


In [7]:
def validate_split(X_train, X_test, y_train, y_test, config):
    """
    Quick validation checks for preprocessed dataset.
    """
    print("Train shape:", X_train.shape)
    print("Test shape:", X_test.shape)

    # Target distribution check
    print("Target distribution in train:\n", y_train.value_counts(normalize=True))
    print("Target distribution in test:\n", y_test.value_counts(normalize=True))

    # Scaling check (only for numeric columns in config)
    for col in config["scale_cols"]:
        print(f"Scaled {col} range (train): {X_train[col].min()} to {X_train[col].max()}")

    # Missing values check
    print("Missing values in train:", X_train.isnull().sum().sum())
    print("Missing values in test:", X_test.isnull().sum().sum())

    # Encoding check
    for col in config["encode_cols"]:
        if col in X_train.columns:
            print(f"Unique values in {col} (train):", X_train[col].unique())


# Validation Helper

We use a helper function `validate_split()` to confirm preprocessing worked correctly:

- **Train/Test shape** -> verifies split size matches config.
- **Target distribution** -> ensures stratification preserved class balance.
- **Scaled range** -> checks numeric scaling (e.g., values between 0–1 for MinMax).
- **Missing values** -> confirms no nulls remain.
- **Encoding check** -> verifies categorical columns were converted to numeric codes.

This makes validation **reusable and consistent** across CSV, Excel, and JSON datasets.




In [10]:
# Load dataset
file_path = "modis_2024_India.csv"  # could be CSV, Excel, or JSON
df = load(file_path)

# Auto-select config
config = get_config(file_path, configs)

# Preprocess
X_train, X_test, y_train, y_test = preprocess_data(df, config)

# Validate
validate_split(X_train, X_test, y_train, y_test, config)



Loaded dataset with shape (74029, 15) from modis_2024_India.csv
Preprocessed dataset: (74029, 15) rows, 15 columns
Train shape: (59223, 14)
Test shape: (14806, 14)
Target distribution in train:
 confidence
67    0.027608
66    0.027388
65    0.026729
63    0.026510
64    0.026493
        ...   
4     0.000152
6     0.000101
2     0.000051
3     0.000051
1     0.000051
Name: proportion, Length: 101, dtype: float64
Target distribution in test:
 confidence
67    0.027624
66    0.027354
65    0.026746
63    0.026543
64    0.026476
        ...   
7     0.000203
4     0.000135
6     0.000135
5     0.000135
2     0.000068
Name: proportion, Length: 99, dtype: float64
Scaled frp range (train): 0.0 to 0.673945673108853
Missing values in train: 0
Missing values in test: 0
Unique values in satellite (train): [1 0]


# Validation Results

| Check                        | Result                          |
|------------------------------|---------------------------------|
| Loaded dataset shape         | (74029, 15)                     |
| Preprocessed dataset shape   | (74029, 15)                     |
| Train shape                  | (59223, 14)                     |
| Test shape                   | (14806, 14)                     |
| Target distribution (train)  | 101 unique confidence levels     |
| Target distribution (test)   | 99 unique confidence levels      |
| Scaled FRP range (train)     | 0.0 to 0.6739                   |
| Missing values (train)       | 0                               |
| Missing values (test)        | 0                               |
| Unique values in satellite   | [0, 1]                          |
